In [ ]:
from dotenv import load_dotenv

load_dotenv()

# Intialize Model and Tools

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from tavily import TavilyClient
from typing import Dict, Any
from pprint import pprint

model = init_chat_model(
    model='gpt-5-nano',
    temperature=0.8,
)

# Web Searching
tavily_client = TavilyClient()

@tool(description="Search the web for information")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

# Stream Output instead of waiting
def stream_output(agent, user_msg, config: Dict[str, Dict[str, str]]):
    for token, metadata in agent.stream(
        {"messages": user_msg},
        config=config,
        stream_mode="messages"
    ):
        if token.content:
            print(token.content, end="", flush=True)


# Create Agent with Memory

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

#agent = create_agent(model='gpt-5-nano')
agent = create_agent(
    model=model,
    tools=[web_search],
    checkpointer=InMemorySaver(),
    system_prompt="You are a Chef that works at high-end restaurents and you have a list of available ingredients, your task is to fetch recipes from the internet if you don't" \
    "already have them in your memory, recipes that can be made using our current list of available ingredients",
)

In [ ]:
user_message: list[HumanMessage] = [
    "Tell me about yourself", 
]
config = {"configurable": {"thread_id": "1"}}

stream_output(agent, user_message, config)

## Adding functionality for multimodal messages

In [ ]:
import base64
import cv2
import matplotlib.pyplot as plt

with open("list.png", "rb") as f:
    img_64 = base64.b64encode(f.read()).decode("utf-8")

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "These are all the ingredients I have, keep them in memory"},
    {"type": "image", "base64": img_64, "mime_type": "image/png"},
])

#Display Image
cv_img = cv2.imread("list.png")
plt.imshow(cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
stream_output(agent, multimodal_question, config)

# Test Run

In [ ]:
agent_response = agent.stream(
    {"messages": [HumanMessage(content="Give me a quick recipe for any kind of snacks with step by step instructions")]},
    stream_mode="messages",
    config = {"configurable": {"thread_id": "1"}}
)

for token, metadata in agent_response:
    if token.content:
        print(token.content, end="", flush=True)